In [2]:
%%writefile 05_zonal_eval.py
"""
Lead-matched evaluation of the zonal TFT, aggregated to statewide.

Mirrors the statewide protocol exactly: 5 passes (one per forecast lead),
whole-column weather swap from the archive boundary onward, interpolate(limit=6)
gap handling, MAPE by day on the summed 11-zone forecast, seasonal breakdown.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# ----------------------------- configuration -----------------------------
ROOT      = Path("/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
FEATURES  = ROOT / "zonal_features.parquet"
LEADS     = ROOT / "weather_leads_zonal.parquet"
CKPT      = ROOT / "checkpoints_zonal/zonal_tft_best.ckpt"
BASE_JSON = ROOT / "metrics_lead_matched.json"      # statewide baseline
OUT_JSON  = ROOT / "metrics_zonal_lead_matched.json"

TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2023-01-01 05:00"
TEST_START  = "2024-01-23 12:00"

ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 256
SEED = 42

ZONE_WEATHER = {
    "WEST": "A_WEST", "GENESE": "B_GENESE", "CENTRL": "C_CENTRL",
    "NORTH": "D_NORTH", "MHK VL": "E_MHKVL", "CAPITL": "F_CAPITL",
    "HUD VL": "G_HUDVL", "MILLWD": "G_HUDVL", "DUNWOD": "J_NYC",
    "N.Y.C.": "J_NYC", "LONGIL": "K_LONGIL",
}
LEAN_VARS = ["temperature_2m", "apparent_temperature", "relative_humidity_2m",
             "wind_speed_10m", "shortwave_radiation", "cloud_cover"]
UNKNOWN_REALS = ["demand", "demand_lag24", "demand_lag168",
                 "demand_roll24_mean", "demand_roll168_mean",
                 "demand_roll24_std"]
KNOWN_REALS = LEAN_VARS + ["temp_vshape", "time_idx"]
KNOWN_CATS = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]


def load_base():
    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS:
        df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()
    return df.sort_values(["zone", "time_idx"]).reset_index(drop=True)


def swap_lead(base, leads, d):
    """Copy of base with the 6 weather vars swapped to previous_day{d},
    leak-free fallback chain, temp_vshape recomputed."""
    out = base.copy()
    boundary = pd.Timestamp(TEST_START)
    leak_fallbacks = 0
    for zone, wkey in ZONE_WEATHER.items():
        mask = out["zone"] == zone
        utc = out.loc[mask, "utc"]
        post = utc >= boundary
        for v in LEAN_VARS:
            s = utc.map(leads[f"{v}_prev_day{d}__{wkey}"]).interpolate(limit=6)
            s5 = utc.map(leads[f"{v}_prev_day5__{wkey}"]).interpolate(limit=6)
            s = s.fillna(s5)
            leak_fallbacks += int((s.isna() & post).sum())
            s = s.fillna(out.loc[mask, v])          # observed: pre-boundary
            out.loc[mask, v] = s.values
    out["temp_vshape"] = (out["temperature_2m"] - 14.0).abs()
    print(f"  post-boundary observed-fallback cells: {leak_fallbacks}"
          f"{'  <-- INVESTIGATE' if leak_fallbacks > 0 else ''}")
    return out


def build_training_ds(df):
    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    train_df = df[df["time_idx"] < val_idx]
    return TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )


def predict_pass(model, training_ds, df, test_start_idx):
    test = TimeSeriesDataSet.from_dataset(
        training_ds, df, min_prediction_idx=test_start_idx,
        stop_randomization=True)
    dl = test.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    preds = model.predict(dl, mode="prediction", return_y=True,
                          return_index=True,
                          trainer_kwargs={"accelerator": "auto",
                                          "enable_progress_bar": True})
    y_hat = preds.output.cpu().numpy()
    y_true = preds.y[0].cpu().numpy()
    tidx = preds.index["time_idx"].values
    return y_hat, y_true, tidx


def aggregate_statewide(y_hat, y_true, tidx):
    """Sum the 11 zones per prediction origin; drop incomplete windows."""
    uniq, inv, counts = np.unique(tidx, return_inverse=True, return_counts=True)
    sum_hat = np.zeros((len(uniq), y_hat.shape[1]))
    sum_true = np.zeros_like(sum_hat)
    np.add.at(sum_hat, inv, y_hat)
    np.add.at(sum_true, inv, y_true)
    complete = counts == 11
    if (~complete).any():
        print(f"  dropped {(~complete).sum()} incomplete windows "
              f"(zones present != 11)")
    return sum_hat[complete], sum_true[complete], uniq[complete]


def main():
    pl.seed_everything(SEED)
    df = load_base()
    leads = pd.read_parquet(LEADS)
    leads.index = pd.to_datetime(leads.index)
    if getattr(leads.index, "tz", None) is not None:
        leads.index = leads.index.tz_localize(None)

    test_start_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    print(f"test_start_idx: {test_start_idx}, "
          f"rows: {len(df):,}, zones: {df['zone'].nunique()}")

    training_ds = build_training_ds(df)
    model = TemporalFusionTransformer.load_from_checkpoint(str(CKPT))

    # statewide baseline day MAPEs, from the artifact if available
    baseline = None
    if BASE_JSON.exists():
        b = json.loads(BASE_JSON.read_text())
        baseline = {int(k): v for k, v in b["day_mape"].items()}
        print(f"baseline loaded: overall {b['overall_mape']}%")

    # month lookup on the shared backbone (any single zone)
    one = df[df["zone"] == df["zone"].iloc[0]]
    month_arr = np.full(int(df["time_idx"].max()) + DECODER_LEN + 2, -1)
    month_arr[one["time_idx"].values] = one["utc"].dt.month.values

    day_mape, composite = {}, []
    for d in range(1, 6):
        print(f"\n=== Lead {d}: swap -> predict -> aggregate ===")
        df_d = swap_lead(df, leads, d)
        y_hat, y_true, tidx = predict_pass(model, training_ds, df_d,
                                           test_start_idx)
        sh, st, ut = aggregate_statewide(y_hat, y_true, tidx)
        s, e = (d - 1) * 24, d * 24
        ape = np.abs(st[:, s:e] - sh[:, s:e]) / np.clip(st[:, s:e], 1e-6, None)
        day_mape[d] = float(ape.mean() * 100)
        base_str = f" (baseline {baseline[d]:.2f}%)" if baseline else ""
        print(f"Day {d} zonal-aggregated MAPE: {day_mape[d]:.2f}%{base_str}")
        composite.append((ape, month_arr[ut[:, None] + np.arange(s, e)[None, :]]))

    all_ape = np.concatenate([a.ravel() for a, _ in composite])
    all_mon = np.concatenate([m.ravel() for _, m in composite])
    overall = float(all_ape.mean() * 100)

    season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring",
                  4: "Spring", 5: "Spring", 6: "Summer", 7: "Summer",
                  8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
    seasons = {}
    for sn in ["Winter", "Spring", "Summer", "Fall"]:
        mask = np.isin(all_mon, [m for m, x in season_map.items() if x == sn])
        seasons[sn] = float(all_ape[mask].mean() * 100) if mask.any() else None

    print("\n============ ZONAL LEAD-MATCHED RESULTS ============")
    for d in range(1, 6):
        base_str = f"   (statewide baseline {baseline[d]:.2f}%)" if baseline else ""
        print(f"Day {d}: {day_mape[d]:.2f}%{base_str}")
    print(f"Overall: {overall:.2f}%   (statewide baseline 3.96%)")
    print("\nSeasonal breakdown:")
    for sn, v in seasons.items():
        print(f"  {sn}: {v:.2f}%")

    out = {"day_mape": {str(k): round(v, 3) for k, v in day_mape.items()},
           "overall_mape": round(overall, 3),
           "seasonal_mape": {k: (round(v, 3) if v else None)
                             for k, v in seasons.items()},
           "checkpoint": str(CKPT),
           "note": "epoch-0 checkpoint, LR=1e-3 run; retrain at 3e-4 pending"}
    OUT_JSON.write_text(json.dumps(out, indent=2))
    print(f"\nSaved -> {OUT_JSON}")


if __name__ == "__main__":
    main()

Overwriting 05_zonal_eval.py


In [3]:
!tail -20 eval_zonal.log

tail: cannot open 'eval_zonal.log' for reading: No such file or directory


In [4]:
import subprocess, sys

with open("eval_zonal.log", "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched")

launched


In [4]:
import subprocess, sys

log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched ->", log_path)

launched -> /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log


In [6]:
import subprocess

# See what's running
print(subprocess.run(["pgrep", "-af", "05_zonal_eval"],
                     capture_output=True, text=True).stdout)

# Kill all instances
subprocess.run(["pkill", "-f", "05_zonal_eval"])
print("killed")

65012 /opt/app-root/bin/python3 05_zonal_eval.py
65327 /opt/app-root/bin/python3 05_zonal_eval.py

killed


In [2]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Traceback (most recent call last):
  File "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_zonal_eval.py", line 14, in <module>
    import lightning.pytorch as pl
ModuleNotFoundError: No module named 'lightning'


In [6]:
import sys
!{sys.executable} -m pip install lightning pytorch-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/848.6 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 13.5 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.5 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 311.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import subprocess
print(subprocess.run(["pgrep", "-af", "05_zonal_eval"],
                     capture_output=True, text=True).stdout or "NOT RUNNING")

866 /opt/app-root/bin/python3 05_zonal_eval.py



In [6]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [1]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [2]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [3]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [4]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [5]:
import subprocess
print(subprocess.run(["pgrep", "-af", "05_zonal_eval"],
                     capture_output=True, text=True).stdout or "NOT RUNNING")
!nvidia-smi

NOT RUNNING
Mon Jul 13 19:45:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:B8:00.0 Off |                    0 |
| N/A   44C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

In [6]:
%%writefile 05_zonal_eval.py
"""
Lead-matched evaluation of the zonal TFT, aggregated to statewide.
v2: trims to test region before prediction (OOM fix), BATCH=128.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

ROOT      = Path("/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
FEATURES  = ROOT / "zonal_features.parquet"
LEADS     = ROOT / "weather_leads_zonal.parquet"
CKPT      = ROOT / "checkpoints_zonal/zonal_tft_best.ckpt"
BASE_JSON = ROOT / "metrics_lead_matched.json"
OUT_JSON  = ROOT / "metrics_zonal_lead_matched.json"

TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2023-01-01 05:00"
TEST_START  = "2024-01-23 12:00"

ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128                                  # CHANGE 1: was 256
SEED = 42

ZONE_WEATHER = {
    "WEST": "A_WEST", "GENESE": "B_GENESE", "CENTRL": "C_CENTRL",
    "NORTH": "D_NORTH", "MHK VL": "E_MHKVL", "CAPITL": "F_CAPITL",
    "HUD VL": "G_HUDVL", "MILLWD": "G_HUDVL", "DUNWOD": "J_NYC",
    "N.Y.C.": "J_NYC", "LONGIL": "K_LONGIL",
}
LEAN_VARS = ["temperature_2m", "apparent_temperature", "relative_humidity_2m",
             "wind_speed_10m", "shortwave_radiation", "cloud_cover"]
UNKNOWN_REALS = ["demand", "demand_lag24", "demand_lag168",
                 "demand_roll24_mean", "demand_roll168_mean",
                 "demand_roll24_std"]
KNOWN_REALS = LEAN_VARS + ["temp_vshape", "time_idx"]
KNOWN_CATS = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]


def load_base():
    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS:
        df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()
    return df.sort_values(["zone", "time_idx"]).reset_index(drop=True)


def swap_lead(base, leads, d):
    out = base.copy()
    boundary = pd.Timestamp(TEST_START)
    leak_fallbacks = 0
    for zone, wkey in ZONE_WEATHER.items():
        mask = out["zone"] == zone
        utc = out.loc[mask, "utc"]
        post = utc >= boundary
        for v in LEAN_VARS:
            s = utc.map(leads[f"{v}_prev_day{d}__{wkey}"]).interpolate(limit=6)
            s5 = utc.map(leads[f"{v}_prev_day5__{wkey}"]).interpolate(limit=6)
            s = s.fillna(s5)
            leak_fallbacks += int((s.isna() & post).sum())
            s = s.fillna(out.loc[mask, v])
            out.loc[mask, v] = s.values
    out["temp_vshape"] = (out["temperature_2m"] - 14.0).abs()
    print(f"  post-boundary observed-fallback cells: {leak_fallbacks}"
          f"{'  <-- INVESTIGATE' if leak_fallbacks > 0 else ''}", flush=True)
    return out


def build_training_ds(df):
    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    train_df = df[df["time_idx"] < val_idx]
    return TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )


def predict_pass(model, training_ds, df, test_start_idx):
    test = TimeSeriesDataSet.from_dataset(
        training_ds, df, min_prediction_idx=test_start_idx,
        stop_randomization=True)
    dl = test.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    preds = model.predict(dl, mode="prediction", return_y=True,
                          return_index=True,
                          trainer_kwargs={"accelerator": "auto",
                                          "enable_progress_bar": False})
    y_hat = preds.output.cpu().numpy()
    y_true = preds.y[0].cpu().numpy()
    tidx = preds.index["time_idx"].values
    return y_hat, y_true, tidx


def aggregate_statewide(y_hat, y_true, tidx):
    uniq, inv, counts = np.unique(tidx, return_inverse=True, return_counts=True)
    sum_hat = np.zeros((len(uniq), y_hat.shape[1]))
    sum_true = np.zeros_like(sum_hat)
    np.add.at(sum_hat, inv, y_hat)
    np.add.at(sum_true, inv, y_true)
    complete = counts == 11
    if (~complete).any():
        print(f"  dropped {(~complete).sum()} incomplete windows", flush=True)
    return sum_hat[complete], sum_true[complete], uniq[complete]


def main():
    pl.seed_everything(SEED)
    df = load_base()
    leads = pd.read_parquet(LEADS)
    leads.index = pd.to_datetime(leads.index)
    if getattr(leads.index, "tz", None) is not None:
        leads.index = leads.index.tz_localize(None)

    test_start_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    print(f"test_start_idx: {test_start_idx}, rows: {len(df):,}, "
          f"zones: {df['zone'].nunique()}", flush=True)

    training_ds = build_training_ds(df)
    model = TemporalFusionTransformer.load_from_checkpoint(str(CKPT))

    baseline = None
    if BASE_JSON.exists():
        b = json.loads(BASE_JSON.read_text())
        baseline = {int(k): v for k, v in b["day_mape"].items()}
        print(f"baseline loaded: overall {b['overall_mape']}%", flush=True)

    # month lookup built from the full frame BEFORE trimming
    one = df[df["zone"] == df["zone"].iloc[0]]
    month_arr = np.full(int(df["time_idx"].max()) + DECODER_LEN + 2, -1)
    month_arr[one["time_idx"].values] = one["utc"].dt.month.values

    # CHANGE 2: trim to test region + encoder warm-up before prediction.
    # training_ds already holds the normalization stats; the predict passes
    # only need rows the test windows can actually touch.
    df = df[df["time_idx"] >= test_start_idx - ENCODER_LEN - 1].copy()
    print(f"trimmed to test region: {len(df):,} rows", flush=True)

    day_mape, composite = {}, []
    for d in range(1, 6):
        print(f"\n=== Lead {d}: swap -> predict -> aggregate ===", flush=True)
        df_d = swap_lead(df, leads, d)
        y_hat, y_true, tidx = predict_pass(model, training_ds, df_d,
                                           test_start_idx)
        del df_d                                     # CHANGE 3: free between passes
        sh, st, ut = aggregate_statewide(y_hat, y_true, tidx)
        s, e = (d - 1) * 24, d * 24
        ape = np.abs(st[:, s:e] - sh[:, s:e]) / np.clip(st[:, s:e], 1e-6, None)
        day_mape[d] = float(ape.mean() * 100)
        base_str = f" (baseline {baseline[d]:.2f}%)" if baseline else ""
        print(f"Day {d} zonal-aggregated MAPE: {day_mape[d]:.2f}%{base_str}",
              flush=True)
        composite.append((ape, month_arr[ut[:, None] + np.arange(s, e)[None, :]]))

    all_ape = np.concatenate([a.ravel() for a, _ in composite])
    all_mon = np.concatenate([m.ravel() for _, m in composite])
    overall = float(all_ape.mean() * 100)

    season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring",
                  4: "Spring", 5: "Spring", 6: "Summer", 7: "Summer",
                  8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
    seasons = {}
    for sn in ["Winter", "Spring", "Summer", "Fall"]:
        mask = np.isin(all_mon, [m for m, x in season_map.items() if x == sn])
        seasons[sn] = float(all_ape[mask].mean() * 100) if mask.any() else None

    print("\n============ ZONAL LEAD-MATCHED RESULTS ============", flush=True)
    for d in range(1, 6):
        base_str = f"   (statewide baseline {baseline[d]:.2f}%)" if baseline else ""
        print(f"Day {d}: {day_mape[d]:.2f}%{base_str}")
    print(f"Overall: {overall:.2f}%   (statewide baseline 3.96%)")
    print("\nSeasonal breakdown:")
    for sn, v in seasons.items():
        print(f"  {sn}: {v:.2f}%")

    out = {"day_mape": {str(k): round(v, 3) for k, v in day_mape.items()},
           "overall_mape": round(overall, 3),
           "seasonal_mape": {k: (round(v, 3) if v else None)
                             for k, v in seasons.items()},
           "checkpoint": str(CKPT),
           "note": "epoch-0 checkpoint, LR=1e-3 run; retrain at 3e-4 pending"}
    OUT_JSON.write_text(json.dumps(out, indent=2))
    print(f"\nSaved -> {OUT_JSON}", flush=True)


if __name__ == "__main__":
    main()

Overwriting 05_zonal_eval.py


In [7]:
!tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automat

In [8]:
import subprocess
print(subprocess.run(["pgrep", "-af", "05_zonal_eval"],
                     capture_output=True, text=True).stdout or "NOT RUNNING")
!nvidia-smi

NOT RUNNING
Mon Jul 13 19:48:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:B8:00.0 Off |                    0 |
| N/A   44C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

In [9]:
!ps aux --sort=-%mem | grep -E "python|jupyter" | grep -v grep

1000950+      50  4.2  0.0 5463336 318468 ?      Sl   18:06   4:25 /opt/app-root/bin/python3 /opt/app-root/bin/jupyter-lab --ServerApp.root_dir=/opt/app-root/src --ServerApp.port=8888 --ServerApp.token='' --ServerApp.password='' --ServerApp.base_url=/notebook/student-notebooks/dsteam1 --ServerApp.quit_button=False --ServerApp.tornado_settings={"user":"rgiffen","hub_host":"https://odh-dashboard-opendatahub.apps.polar.cair.mun.ca","hub_prefix":"/projects/student-notebooks"} --ServerApp.ip= --ServerApp.allow_origin=* --ServerApp.open_browser=False
1000950+    1247  4.3  0.0 6556176 278144 ?      Sl   19:43   0:13 /opt/app-root/bin/python3 -m pylsp
1000950+     291  0.0  0.0 852488 76420 ?        Ssl  18:56   0:02 /opt/app-root/bin/python3 -m ipykernel_launcher -f /opt/app-root/src/.local/share/jupyter/runtime/kernel-74a6e41d-8b7f-4c1a-a69b-536d669fb5ae.json
1000950+     293  0.0  0.0 760336 74792 ?        Ssl  18:56   0:02 /opt/app-root/bin/python3 -m ipykernel_launcher -f /opt/app-root/s

In [10]:
!free -g
!cat /sys/fs/cgroup/memory.max 2>/dev/null || cat /sys/fs/cgroup/memory/memory.limit_in_bytes 2>/dev/null

               total        used        free      shared  buff/cache   available
Mem:            1007          87         623           0         303         920
Swap:              0           0           0


68719476736


In [11]:
!grep -n "trimmed to test region" /opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_zonal_eval.py

150:    print(f"trimmed to test region: {len(df):,} rows", flush=True)


In [15]:
import subprocess, sys
log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched")

launched


In [16]:
!sleep 60 && tail -30 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log && pgrep -af 05_zonal_eval

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, whi

In [1]:
!tail -40 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, whi

In [4]:
!pgrep -af 05_zonal_eval

In [5]:
!pgrep -af 05_zonal_eval && ps -o pid,etime,rss,cmd -p $(pgrep -f 05_zonal_eval)

7150 /usr/bin/sh -c pgrep -af 05_zonal_eval && ps -o pid,etime,rss,cmd -p $(pgrep -f 05_zonal_eval)
    PID     ELAPSED   RSS CMD
   7150       00:00  3072 ps -o pid,etime,rss,cmd -p 7150


In [6]:
# When did it die? (last write to the log)
!ls -l --time-style=full-iso /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

# Suspect 1: OOM killer — this counter increments every time the cgroup kills something
!cat /sys/fs/cgroup/memory.events 2>/dev/null
!cat /sys/fs/cgroup/memory.peak 2>/dev/null

# Suspect 2: the pod restarted again while you were out
# (if lightning is missing, the pod recycled and killed everything with it)
import importlib.util
print("lightning installed:", importlib.util.find_spec("lightning") is not None)
!uptime

-rw-r--r--. 1 1000950000 1000950000 1996 2026-07-13 19:54:17.925988974 +0000 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log


low 0
high 0
max 0
oom 0
oom_kill 0
oom_group_kill 0


4034281472


lightning installed: True
 23:32:58 up 38 days,  8:28,  0 users,  load average: 5.19, 5.77, 7.59


In [7]:
import subprocess, sys
log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched, detached")

launched, detached


In [8]:
path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_zonal_eval.py"
src = open(path).read()
old = 'composite.append((ape, month_arr[ut[:, None] + np.arange(s, e)[None, :]]))'
new = old + '''
        Path(ROOT / "metrics_zonal_partial.json").write_text(
            json.dumps({str(k): round(v, 3) for k, v in day_mape.items()},
                       indent=2))'''
assert src.count(old) == 1
open(path, "w").write(src.replace(old, new))
print("patched")

patched


In [ ]:
import subprocess, sys
log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched, detached")

In [7]:
import subprocess, sys
log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched, detached")

launched, detached


In [8]:
!pgrep -af 05_zonal_eval

1990 /opt/app-root/bin/python3 05_zonal_eval.py


In [1]:
!tail -15 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics

In [2]:
!cat /opt/app-root/src/Forecasting-Energy-Demand/Sangar/metrics_zonal_partial.json

cat: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/metrics_zonal_partial.json: No such file or directory


In [6]:
!tail -20 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, whi

In [9]:
!ps -o pid,sid,etime,rss,cmd -p $(pgrep -f 05_zonal_eval)

    PID     SID     ELAPSED   RSS CMD
   2557    2557       00:00  3072 ps -o pid,sid,etime,rss,cmd -p 2557


In [10]:
!tail -50 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log
!cat /sys/fs/cgroup/memory.events
!cat /sys/fs/cgroup/memory.peak
!uptime

Seed set to 42
test_start_idx: 75080, rows: 1,052,788, zones: 11
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
baseline loaded: overall 3.956%
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, whi

low 0
high 0
max 0
oom 0
oom_kill 0
oom_group_kill 0


1149292544


 07:28:27 up 38 days, 16:23,  0 users,  load average: 8.55, 8.18, 7.29


In [11]:
%%writefile 05_zonal_eval.py
"""
Lead-matched evaluation of the zonal TFT, aggregated to statewide.
v3: RESUMABLE. Each lead's APE matrix is saved to zonal_lead{d}.npz on
completion; finished leads are skipped on relaunch. Survives pod recycles
by never risking more than one ~13-minute pass.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

ROOT      = Path("/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
FEATURES  = ROOT / "zonal_features.parquet"
LEADS     = ROOT / "weather_leads_zonal.parquet"
CKPT      = ROOT / "checkpoints_zonal/zonal_tft_best.ckpt"
BASE_JSON = ROOT / "metrics_lead_matched.json"
OUT_JSON  = ROOT / "metrics_zonal_lead_matched.json"

TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2023-01-01 05:00"
TEST_START  = "2024-01-23 12:00"

ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128
SEED = 42

ZONE_WEATHER = {
    "WEST": "A_WEST", "GENESE": "B_GENESE", "CENTRL": "C_CENTRL",
    "NORTH": "D_NORTH", "MHK VL": "E_MHKVL", "CAPITL": "F_CAPITL",
    "HUD VL": "G_HUDVL", "MILLWD": "G_HUDVL", "DUNWOD": "J_NYC",
    "N.Y.C.": "J_NYC", "LONGIL": "K_LONGIL",
}
LEAN_VARS = ["temperature_2m", "apparent_temperature", "relative_humidity_2m",
             "wind_speed_10m", "shortwave_radiation", "cloud_cover"]
UNKNOWN_REALS = ["demand", "demand_lag24", "demand_lag168",
                 "demand_roll24_mean", "demand_roll168_mean",
                 "demand_roll24_std"]
KNOWN_REALS = LEAN_VARS + ["temp_vshape", "time_idx"]
KNOWN_CATS = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]


def lead_file(d):
    return ROOT / f"zonal_lead{d}.npz"


def load_base():
    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS:
        df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()
    return df.sort_values(["zone", "time_idx"]).reset_index(drop=True)


def swap_lead(base, leads, d):
    out = base.copy()
    boundary = pd.Timestamp(TEST_START)
    leak_fallbacks = 0
    for zone, wkey in ZONE_WEATHER.items():
        mask = out["zone"] == zone
        utc = out.loc[mask, "utc"]
        post = utc >= boundary
        for v in LEAN_VARS:
            s = utc.map(leads[f"{v}_prev_day{d}__{wkey}"]).interpolate(limit=6)
            s5 = utc.map(leads[f"{v}_prev_day5__{wkey}"]).interpolate(limit=6)
            s = s.fillna(s5)
            leak_fallbacks += int((s.isna() & post).sum())
            s = s.fillna(out.loc[mask, v])
            out.loc[mask, v] = s.values
    out["temp_vshape"] = (out["temperature_2m"] - 14.0).abs()
    print(f"  post-boundary observed-fallback cells: {leak_fallbacks}"
          f"{'  <-- INVESTIGATE' if leak_fallbacks > 0 else ''}", flush=True)
    return out


def build_training_ds(df):
    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    train_df = df[df["time_idx"] < val_idx]
    return TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )


def predict_pass(model, training_ds, df, test_start_idx):
    test = TimeSeriesDataSet.from_dataset(
        training_ds, df, min_prediction_idx=test_start_idx,
        stop_randomization=True)
    dl = test.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    preds = model.predict(dl, mode="prediction", return_y=True,
                          return_index=True,
                          trainer_kwargs={"accelerator": "auto",
                                          "enable_progress_bar": False})
    return (preds.output.cpu().numpy(), preds.y[0].cpu().numpy(),
            preds.index["time_idx"].values)


def aggregate_statewide(y_hat, y_true, tidx):
    uniq, inv, counts = np.unique(tidx, return_inverse=True, return_counts=True)
    sum_hat = np.zeros((len(uniq), y_hat.shape[1]))
    sum_true = np.zeros_like(sum_hat)
    np.add.at(sum_hat, inv, y_hat)
    np.add.at(sum_true, inv, y_true)
    complete = counts == 11
    if (~complete).any():
        print(f"  dropped {(~complete).sum()} incomplete windows", flush=True)
    return sum_hat[complete], sum_true[complete], uniq[complete]


def summarize(baseline):
    """Build final summary from the five saved npz files."""
    day_mape, apes, mons = {}, [], []
    for d in range(1, 6):
        z = np.load(lead_file(d))
        day_mape[d] = float(z["ape"].mean() * 100)
        apes.append(z["ape"].ravel())
        mons.append(z["months"].ravel())
    all_ape, all_mon = np.concatenate(apes), np.concatenate(mons)
    overall = float(all_ape.mean() * 100)

    season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring",
                  4: "Spring", 5: "Spring", 6: "Summer", 7: "Summer",
                  8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
    seasons = {}
    for sn in ["Winter", "Spring", "Summer", "Fall"]:
        mask = np.isin(all_mon, [m for m, x in season_map.items() if x == sn])
        seasons[sn] = float(all_ape[mask].mean() * 100) if mask.any() else None

    print("\n============ ZONAL LEAD-MATCHED RESULTS ============", flush=True)
    for d in range(1, 6):
        base_str = f"   (statewide baseline {baseline[d]:.2f}%)" if baseline else ""
        print(f"Day {d}: {day_mape[d]:.2f}%{base_str}")
    print(f"Overall: {overall:.2f}%   (statewide baseline 3.96%)")
    print("\nSeasonal breakdown:")
    for sn, v in seasons.items():
        print(f"  {sn}: {v:.2f}%")

    out = {"day_mape": {str(k): round(v, 3) for k, v in day_mape.items()},
           "overall_mape": round(overall, 3),
           "seasonal_mape": {k: (round(v, 3) if v else None)
                             for k, v in seasons.items()},
           "checkpoint": str(CKPT),
           "note": "epoch-0 checkpoint, LR=1e-3 run; retrain at 3e-4 pending"}
    OUT_JSON.write_text(json.dumps(out, indent=2))
    print(f"\nSaved -> {OUT_JSON}", flush=True)


def main():
    pl.seed_everything(SEED)

    baseline = None
    if BASE_JSON.exists():
        b = json.loads(BASE_JSON.read_text())
        baseline = {int(k): v for k, v in b["day_mape"].items()}

    todo = [d for d in range(1, 6) if not lead_file(d).exists()]
    done = [d for d in range(1, 6) if lead_file(d).exists()]
    print(f"leads done: {done or 'none'}, todo: {todo or 'none'}", flush=True)

    if todo:
        df = load_base()
        leads = pd.read_parquet(LEADS)
        leads.index = pd.to_datetime(leads.index)
        if getattr(leads.index, "tz", None) is not None:
            leads.index = leads.index.tz_localize(None)

        test_start_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
        training_ds = build_training_ds(df)
        model = TemporalFusionTransformer.load_from_checkpoint(str(CKPT))

        one = df[df["zone"] == df["zone"].iloc[0]]
        month_arr = np.full(int(df["time_idx"].max()) + DECODER_LEN + 2, -1)
        month_arr[one["time_idx"].values] = one["utc"].dt.month.values

        df = df[df["time_idx"] >= test_start_idx - ENCODER_LEN - 1].copy()
        print(f"trimmed to test region: {len(df):,} rows", flush=True)

        for d in todo:
            print(f"\n=== Lead {d}: swap -> predict -> aggregate ===", flush=True)
            df_d = swap_lead(df, leads, d)
            y_hat, y_true, tidx = predict_pass(model, training_ds, df_d,
                                               test_start_idx)
            del df_d
            sh, st, ut = aggregate_statewide(y_hat, y_true, tidx)
            s, e = (d - 1) * 24, d * 24
            ape = np.abs(st[:, s:e] - sh[:, s:e]) / np.clip(st[:, s:e],
                                                            1e-6, None)
            months = month_arr[ut[:, None] + np.arange(s, e)[None, :]]
            np.savez(lead_file(d), ape=ape, months=months)
            base_str = f" (baseline {baseline[d]:.2f}%)" if baseline else ""
            print(f"Day {d} zonal-aggregated MAPE: "
                  f"{float(ape.mean() * 100):.2f}%{base_str}  [saved]",
                  flush=True)

    if all(lead_file(d).exists() for d in range(1, 6)):
        summarize(baseline)
    else:
        print("\nnot all leads complete yet - relaunch to continue", flush=True)


if __name__ == "__main__":
    main()

Overwriting 05_zonal_eval.py


In [1]:
!tail -50 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
leads done: none, todo: [1, 2, 3, 4, 5]
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automaticall

In [15]:
!grep -c "zonal_lead{d}.npz" /opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_zonal_eval.py

2


In [16]:
!ls -l /opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz

ls: cannot access '/opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz': No such file or directory


In [18]:
import subprocess, sys, importlib.util

# 1. reinstall if the recycle wiped packages
if importlib.util.find_spec("lightning") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightning", "pytorch-forecasting"])
    print("packages reinstalled")

# 2. relaunch (v3 skips any banked leads automatically)
log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("relaunched")

packages reinstalled
relaunched



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!ls /opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz 2>/dev/null; pgrep -f 05_zonal_eval > /dev/null && echo RUNNING || echo DEAD; tail -3 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

RUNNING
You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


In [3]:
import os
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
for f in ["zonal_features.parquet", "weather_leads_zonal.parquet",
          "checkpoints_zonal/zonal_tft_best.ckpt", "05_zonal_eval.py",
          "metrics_lead_matched.json"]:
    p = os.path.join(root, f)
    print(f"{'OK ' if os.path.exists(p) else 'MISSING':8} {f}")

# is v3 (resumable) still the version on disk?
print("---")
os.system(f"grep -c 'leads done' {root}/05_zonal_eval.py")

OK       zonal_features.parquet
OK       weather_leads_zonal.parquet
OK       checkpoints_zonal/zonal_tft_best.ckpt
OK       05_zonal_eval.py
OK       metrics_lead_matched.json
---
1


0

In [4]:
import subprocess, sys, importlib.util
if importlib.util.find_spec("lightning") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightning", "pytorch-forecasting"])
    print("packages reinstalled")

log_path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log"
with open(log_path, "w") as log:
    subprocess.Popen([sys.executable, "05_zonal_eval.py"],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("launched")

launched


In [11]:
!ls /opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz 2>/dev/null; pgrep -f 05_zonal_eval >/dev/null && echo RUNNING || echo DEAD; tail -3 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

RUNNING
I0000 00:00:1784634549.221981  506524 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


In [12]:
import subprocess, os
root = "/opt/app-root/src/Forecasting-Energy-Demand"
os.chdir(root)

# Is this a DVC/DagsHub-backed repo, and what remotes exist?
print(subprocess.run(["git", "remote", "-v"], capture_output=True, text=True).stdout)
print("--- dvc remotes ---")
print(subprocess.run(["dvc", "remote", "list"], capture_output=True, text=True).stdout or "no dvc or no remotes")
print("--- are the parquets DVC-tracked? ---")
for f in ["Sangar/zonal_features.parquet", "Sangar/weather_leads_zonal.parquet"]:
    print(f, "->", "tracked" if os.path.exists(f + ".dvc") else "NOT in dvc")

myrepo	https://Sangi2805@github.com/Sangi2805/Forecasting-Energy-Demand.git (fetch)
myrepo	https://Sangi2805@github.com/Sangi2805/Forecasting-Energy-Demand.git (push)
origin	https://github.com/txdoan-cpu/Forecasting-Energy-Demand.git (fetch)
origin	https://github.com/txdoan-cpu/Forecasting-Energy-Demand.git (push)
sangar	https://github.com/Sangi2805/Forecasting-Energy-Demand.git (fetch)
sangar	https://github.com/Sangi2805/Forecasting-Energy-Demand.git (push)

--- dvc remotes ---


FileNotFoundError: [Errno 2] No such file or directory: 'dvc'

In [10]:
import os, subprocess
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

# sizes of the three files worth sharing
for f in ["zonal_features.parquet", "weather_observed_zonal.parquet",
          "weather_leads_zonal.parquet"]:
    p = os.path.join(root, f)
    print(f"{os.path.getsize(p)/1e6:7.1f} MB  {f}")

# is this shared storage? whose home is this, and is there a shared mount?
print("---")
print("whoami:", subprocess.run(["whoami"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["df", "-h", root], capture_output=True, text=True).stdout)

   66.8 MB  zonal_features.parquet
   13.4 MB  weather_observed_zonal.parquet
   11.9 MB  weather_leads_zonal.parquet
---
whoami: 1000950000
Filesystem      Size  Used Avail Use% Mounted on
/dev/rbd1        20G  5.4G   15G  28% /opt/app-root/src



In [13]:
!ls /opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz 2>/dev/null; pgrep -f 05_zonal_eval >/dev/null && echo RUNNING || echo DEAD; tail -2 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

RUNNING
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


In [14]:
import subprocess
# how long has it actually been alive?
pid = subprocess.run(["pgrep", "-f", "05_zonal_eval"], capture_output=True, text=True).stdout.strip()
print("PID:", pid)
print(subprocess.run(["ps", "-o", "etime=,%cpu=,rss=", "-p", pid],
                     capture_output=True, text=True).stdout)
# has ANY lead banked yet?
import glob
print("banked:", glob.glob("/opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz") or "none")

PID: 

banked: none


In [15]:
!tail -25 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_zonal.log

Seed set to 42
leads done: none, todo: [1, 2, 3, 4, 5]
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
trimmed to test region: 228,767 rows

=== Lead 1: swap -> predict -> aggregate ===
  post-boundary observed-fallback cells: 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automaticall

In [16]:
import subprocess, glob, time

# 1. Is the process I just launched actually alive right now?
pid = subprocess.run(["pgrep","-f","05_zonal_eval"], capture_output=True, text=True).stdout.strip()
print("PID now:", pid or "DEAD")
if pid:
    print(subprocess.run(["ps","-o","etime=,%cpu=,rss=","-p",pid], capture_output=True, text=True).stdout)

# 2. What's on the GPU? (stale process from last week = contention)
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory",
                      "--format=csv"], capture_output=True, text=True).stdout)

# 3. banked leads
print("banked:", glob.glob("/opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_lead*.npz") or "none")

PID now: DEAD
pid, used_gpu_memory [MiB]
4381, 20402 MiB

banked: none


In [17]:
import subprocess
subprocess.run(["kill", "-9", "4381"])
import time; time.sleep(5)
# confirm the GPU is freed
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)

pid, used_gpu_memory [MiB]



In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)

In [18]:
import pandas as pd
df = pd.read_parquet("/opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_features.parquet")
print("columns:", list(df.columns))
print("\nzone dtype:", df["zone"].dtype)
print("unique zones:", sorted(df["zone"].unique()))
print("count:", df["zone"].nunique())

columns: ['utc', 'zone', 'demand', 'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'wind_speed_10m', 'shortwave_radiation', 'cloud_cover', 'temp_vshape', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'demand_lag24', 'demand_lag168', 'demand_roll24_mean', 'demand_roll168_mean', 'demand_roll24_std', 'time_idx']

zone dtype: object
unique zones: ['CAPITL', 'CENTRL', 'DUNWOD', 'GENESE', 'HUD VL', 'LONGIL', 'MHK VL', 'MILLWD', 'N.Y.C.', 'NORTH', 'WEST']
count: 11


In [19]:
import os
p = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/checkpoints_zonal/zonal_tft_best.ckpt"
print("exists:", os.path.exists(p), "| size:", round(os.path.getsize(p)/1e6, 1), "MB" if os.path.exists(p) else "")

exists: True | size: 5.5 MB


In [20]:
import pytorch_forecasting, lightning, torch, sklearn
print("pytorch_forecasting:", pytorch_forecasting.__version__)
print("lightning:", lightning.__version__)
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)

pytorch_forecasting: 1.8.0
lightning: 2.6.5
torch: 2.7.1+cu128
sklearn: 1.7.2


In [21]:
!pip install -q scikit-learn==1.7.2 lightning==2.6.5 pytorch-forecasting==1.8.0
print("pinned — now RESTART RUNTIME (Runtime menu > Restart) before running the eval")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
pinned — now RESTART RUNTIME (Runtime menu > Restart) before running the eval


In [23]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)

pid, used_gpu_memory [MiB]
33885, 716 MiB
211679, 770 MiB



In [24]:
import subprocess
print(subprocess.run(["ps","-o","pid,user,etime,cmd","-p","33885,211679"],
                     capture_output=True, text=True).stdout)

    PID USER         ELAPSED CMD
  33885 1000950+    10:43:50 /opt/app-root/bin/python3 -m ipykernel_launcher -f /opt/app-root/src/.local/share/jupyter/runtime/kernel-fb009e40-0c7e-44a7-a9b5-999458ad5e69.json
 211679 1000950+    04:12:39 /opt/app-root/bin/python3 -m ipykernel_launcher -f /opt/app-root/src/.local/share/jupyter/runtime/kernel-b53b1e05-0c85-40d9-a07a-42b16cc0c206.json

